# traitement 

In [2]:
pip install jupyter 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# importation des bibliothéque necessaire 

In [3]:
import os
import numpy as np
import librosa
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm.notebook import tqdm
import pickle

# les differentes emotions du dataset

In [4]:
emotions = {
    "01": "neutre",
    "02": "calme",
    "03": "joie",
    "04": "tristesse",
    "05": "colère",
    "06": "peur",
    "07": "dégoût",
    "08": "surprise"
}

In [5]:
print(" emotions : ", emotions)

 emotions :  {'01': 'neutre', '02': 'calme', '03': 'joie', '04': 'tristesse', '05': 'colère', '06': 'peur', '07': 'dégoût', '08': 'surprise'}


In [6]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, duration=3)
    
    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc_mean = np.mean(mfcc.T, axis=0)
    mfcc_std = np.std(mfcc.T, axis=0)
    
    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma.T, axis=0)
    
    # Mel Spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    mel_mean = np.mean(mel.T, axis=0)
    
    # ZCR et RMS
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rms = np.mean(librosa.feature.rms(y=y))
    
    # Pitch — nouveau !
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0
    pitch_std = np.std(pitches[pitches > 0]) if np.any(pitches > 0) else 0
    
    # Tempo — nouveau !
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    
    return np.hstack([mfcc_mean, mfcc_std, chroma_mean, mel_mean, 
                      [zcr, rms, pitch_mean, pitch_std, float(tempo)]])

In [7]:
def load_dataset(dataset_path):
    X, y = [], []
    fichiers = []

    for actor in os.listdir(dataset_path):
        actor_path = os.path.join(dataset_path, actor)
        if os.path.isdir(actor_path):
            for fichier in os.listdir(actor_path):
                if fichier.endswith(".wav"):
                    fichiers.append(os.path.join(actor_path, fichier))

    print(f"{len(fichiers)} fichiers trouvés")

    for file_path in tqdm(fichiers, desc="Extraction MFCC"):
        try:
            emotion_code = os.path.basename(file_path).split("-")[2]
            if emotion_code not in emotions:
                continue
            mfcc = extract_features(file_path)
            X.append(mfcc)
            y.append(emotions[emotion_code])
        except Exception as e:
            print(f"Erreur : {e}")

    return np.array(X), np.array(y)

# chargement du dataset

In [8]:
# Change ce chemin par le tien
dataset_path = r"C:\Users\Moham\Downloads\Audio_Song_Actors_01-24"
X, y = load_dataset(dataset_path)
print(f"Dataset : {X.shape[0]} échantillons, {X.shape[1]} features")

1012 fichiers trouvés


Extraction MFCC:   0%|          | 0/1012 [00:00<?, ?it/s]

C:\Users\Moham\AppData\Local\Temp\ipykernel_32548\29530955.py:30: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  [zcr, rms, pitch_mean, pitch_std, float(tempo)]])


Dataset : 1012 échantillons, 225 features


In [9]:
print("Séparation des données en train et test...")

Séparation des données en train et test...


In [ ]:

from sklearn.preprocessing import StandardScaler, LabelEncoder

# Encoder les labels en chiffres
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Normalisation des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modèle plus puissant
model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    max_iter=500,
    random_state=42,
    learning_rate_init=0.001,
    early_stopping=True
)
model.fit(X_train_scaled, y_train_encoded)
print("Modèle entraîné ✅")

NameError: name 'y_train' is not defined

In [ ]:
y_pred_encoded = model.predict(X_test_scaled)

# Reconvertir les chiffres en noms d'émotions
y_pred = le.inverse_transform(y_pred_encoded)
y_test_labels = le.inverse_transform(y_test_encoded)

accuracy = accuracy_score(y_test_labels, y_pred)
print(f"Accuracy : {accuracy * 100:.2f}%")
print(classification_report(y_test_labels, y_pred))

Accuracy : 90.64%
              precision    recall  f1-score   support

       calme       0.95      1.00      0.98        41
      colère       0.97      0.92      0.95        38
        joie       1.00      0.93      0.97        30
      neutre       1.00      1.00      1.00        21
        peur       0.85      0.65      0.73        34
   tristesse       0.76      0.95      0.84        39

    accuracy                           0.91       203
   macro avg       0.92      0.91      0.91       203
weighted avg       0.91      0.91      0.90       203



In [ ]:
import tkinter as tk
from tkinter import filedialog

# Fonction pour ouvrir l'explorateur de fichiers
def choose_audio_file():
    root = tk.Tk()
    root.withdraw()
    file_path = filedialog.askopenfilename(
        title="Choisis un fichier audio",
        filetypes=[("Fichiers WAV", "*.wav")]
    )
    return file_path if file_path else None

# Test sur ta propre voix
audio_file = choose_audio_file()

if audio_file:
    print(f"Fichier sélectionné : {audio_file}")
    features = extract_features(audio_file)
    features_scaled = scaler.transform([features])
    pred_encoded = model.predict(features_scaled)[0]
    prediction = le.inverse_transform([pred_encoded])[0]
    proba = model.predict_proba(features_scaled)[0]

    print(f"\n Émotion détectée : {prediction.upper()}")
    print("\nProbabilités :")
    for classe, prob in sorted(zip(le.classes_, proba), key=lambda x: -x[1]):
        barre = "" * int(prob * 20)
        print(f"  {classe:<12} {barre} {prob*100:.1f}%")
else:
    print("Aucun fichier sélectionné.")

Fichier sélectionné : C:/Users/Moham/Downloads/man laughing; Free Sound Effect - Comment Music (128k).wav


NameError: name 'extract_features' is not defined